### **Key Considerations for Your Dataproc Cluster**

1.  **Cluster Resources:**

    -   **Master:** `n2-standard-4` (4 vCPUs, 16 GB RAM, 32GB disk)
    -   **Workers (2x):** `n2-standard-4` (4 vCPUs, 16 GB RAM, 64GB disk each)
    -   **Total:** 8 worker vCPUs, ~32 GB RAM (excluding master node)
2.  **Dataproc Features Disabled:**

    -   No **autoscaling**, **Metastore**, **advanced execution layer**, **advanced optimizations**
    -   **Storage:** `pd-balanced` (no SSDs, so I/O optimization is crucial)
    -   **Networking:** Internal IP **enabled**
3.  **Optimization Strategy:**

    -   Tune **shuffle partitions**, **broadcast join threshold**, and **storage persistence**
    -   Adjust **parallelism** based on **2 workers x 4 cores**
    -   Avoid **excessive caching** due to **disk-based storage**

In [1]:
from pyspark.sql import SparkSession

In [2]:
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Olist Ecommerce Performance Optimization') \
    .config('spark.executor.memory', '4g') \
    .config('spark.executor.cores', '2') \
    .config('spark.executor.instances', '2') \
    .config('spark.driver.memory', '2g') \
    .config('spark.driver.maxResultSize', '1g') \
    .config('spark.sql.shuffle.partitions', '64') \
    .config('spark.default.parallelism', '64') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.sql.autoBroadcastJoinThreshold', 20 * 1024 * 1024) \
    .config('spark.sql.files.maxPartitionBytes', '64MB') \
    .config('spark.sql.files.openCostInBytes', '2MB') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 13:30:03 INFO SparkEnv: Registering MapOutputTracker
26/04/22 13:30:03 INFO SparkEnv: Registering BlockManagerMaster
26/04/22 13:30:03 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/22 13:30:03 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
# to simplfy I am writing thhe hdfs_path and calling the files in that folder
hdfs_path = '/tmp/brazilian-ecommerce/'

In [4]:
## Reading the customer data file
## With inferSchema as true, Spark automatically detects and assigns appropriate column data types based on the data

customers_df= spark.read.format("csv").option("header","true").option("inferSchema", "true").load(hdfs_path+'olist_customers_dataset.csv')

## Reading the olist_orders_dataset file
orders_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_orders_dataset.csv')

## Reading the olist_order_items_dataset file
order_item_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_items_dataset.csv')

## Reading the order_payments file
payments_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_payments_dataset.csv')

## Reading the olist_order_reviews_dataset file
reviews_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_reviews_dataset.csv')

## Reading the olist_products_dataset file
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema=True)

## Reading the olist_sellers_dataset file
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)

## Reading the olist_geolocation_dataset file
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)

## Reading the olist_orders_dataset file
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)

26/04/22 13:31:14 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/22 13:31:29 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/22 13:31:44 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


## Optimized Join Stragies

In [5]:
!hadoop fs -ls /tmp/olist

Found 11 items
-rw-r--r--   2 root hadoop          0 2026-04-22 13:28 /tmp/olist/_SUCCESS
-rw-r--r--   2 root hadoop    9237063 2026-04-22 13:27 /tmp/olist/part-00000-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop    8504464 2026-04-22 13:27 /tmp/olist/part-00001-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop    7993652 2026-04-22 13:27 /tmp/olist/part-00002-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop    9823474 2026-04-22 13:27 /tmp/olist/part-00003-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop    9733948 2026-04-22 13:28 /tmp/olist/part-00004-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop    8793278 2026-04-22 13:28 /tmp/olist/part-00005-dabc22f8-8568-4bc1-bfe0-c22b3628b6ae-c000.snappy.parquet
-rw-r--r--   2 root hadoop   10122433 2026-04-22 13:28 /tmp/olist/part-00006-dabc22f8-8568-4bc1-bfe0-c22b3

In [8]:
full_orders_df = spark.read.parquet('/tmp/olist/')

In [9]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

1. Broadcast: Used when one table is small. Spark sends (broadcasts) the small table to all executors. 
2. Small table → copied to every node, Large table → stays distributed. Join happens locally → no shuffle

In [12]:
# Broadcast
from pyspark.sql.functions import *

customers_broadcast_df = broadcast(customers_df)
optimized_broadcast_join = full_orders_df.join(customers_broadcast_df,'customer_id')


1. Sort and Merge join: Default join used by Spark for large datasets
2. Shuffle both datasets. Sort them by join key. Merge them

In [13]:
# Sort and Merge join

sorted_customers_df = customers_df.sortWithinPartitions('customer_id')
sorted_orders_df = full_orders_df.sortWithinPartitions('customer_id')


optimized_merge_full_orders_df = sorted_orders_df.join(sorted_customers_df,'customer_id')

### sortWithinPartitions is used to sort data inside each partition, not across the entire DataFrame
### sortWithinPartitions sorts rows locally within each partition without triggering a full shuffle
## repartition used to change the number of partitions in a DataFrame (increase or decrease) with full data redistribution


In [14]:
# Bucket join 

bucketed_customers_df = customers_df.repartition(10,'customer_id')
bucketed_orders_df = full_orders_df.repartition(10,'customer_id')

bucket_join_df = bucketed_orders_df.join(bucketed_customers_df,'customer_id')

In [ ]:
# Skew Join handling

skew_handled_join = full_orders_df.join(customers_df.hint('skew'),'customer_id')